In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from category_encoders import WOEEncoder
import warnings

warnings.filterwarnings('ignore')

print("\n--- KAGGLE FORUM TAKTİĞİ: WoE (Weight of Evidence) KODLAMA ---")

# 1. VERİLERİ YÜKLEME (Gerçek + Sentetik)
train = pd.read_csv("../data/raw/train.csv", index_col='id')
test = pd.read_csv("../data/raw/test.csv", index_col='id')

url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
orig = pd.read_csv(url)
orig = orig.rename(columns={'customerID': 'id'}).set_index('id')

y_train = train['Churn'].map({'Yes': 1, 'No': 0})
train.drop('Churn', axis=1, inplace=True)

y_orig = orig['Churn'].map({'Yes': 1, 'No': 0})
orig.drop('Churn', axis=1, inplace=True)

X_train_full = pd.concat([train, orig])
y_train_full = pd.concat([y_train, y_orig])

df_all = pd.concat([X_train_full, test], keys=['train', 'test'])
df_all['TotalCharges'] = pd.to_numeric(df_all['TotalCharges'], errors='coerce').fillna(0)

# --- REKOR KIRAN SİHİRLİ ÖZELLİKLERİMİZ ---
df_all['TotalCharges_Diff'] = df_all['TotalCharges'] - (df_all['MonthlyCharges'] * df_all['tenure'])
ek_hizmetler = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df_all['Total_Extra_Services'] = (df_all[ek_hizmetler] == 'Yes').sum(axis=1).astype(str) # Kategoriye çeviriyoruz
contract_map = {'Month-to-month': 1, 'One year': 12, 'Two year': 24}
df_all['Contract_Months'] = df_all['Contract'].map(contract_map)
df_all['Tenure_Contract_Ratio'] = df_all['tenure'] / df_all['Contract_Months']
df_all['Auto_Payment'] = df_all['PaymentMethod'].isin(['Bank transfer (automatic)', 'Credit card (automatic)']).astype(str)
df_all['Has_Family'] = ((df_all['Partner'] == 'Yes') | (df_all['Dependents'] == 'Yes')).astype(str)

# -------------------------------------------------------------------
# KANIT AĞIRLIĞI (WoE) İÇİN SAYISAL VERİLERİ KUTULARA BÖLME (BINNING)
# -------------------------------------------------------------------
print("Sayısal veriler parçalanıyor (QCut Binning)...")
num_cols_to_bin = ['tenure', 'MonthlyCharges', 'TotalCharges', 'TotalCharges_Diff']

for col in num_cols_to_bin:
    # Sayısal verileri 10 eşit parçaya bölüp kategorik (string) hale getiriyoruz
    df_all[f'{col}_Binned'] = pd.qcut(df_all[col], q=10, duplicates='drop').astype(str)

# Orijinal sayısal sütunları düşürebiliriz çünkü binned versiyonlarına WoE uygulayacağız
df_all.drop(columns=num_cols_to_bin, inplace=True)

# Geriye kalan tüm sütunlar artık kategorik veya Binned (Kutulanmış)
cat_cols = df_all.columns.tolist()

# Veriyi tekrar ayır
X_train_final = df_all.xs('train')
X_test_final = df_all.xs('test')

print(f"Toplam {len(cat_cols)} sütuna WoE Encode uygulanacak. Eğitim Başlıyor...\n")

# --- XGBOOST EĞİTİMİ VE WOE ENCODING (Sızıntıyı Önlemek İçin Fold İçinde) ---
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X_train_final))
test_preds = np.zeros(len(X_test_final))

for fold, (train_idx, valid_idx) in enumerate(skf.split(X_train_final, y_train_full)):
    X_tr, y_tr = X_train_final.iloc[train_idx].copy(), y_train_full.iloc[train_idx]
    X_va, y_va = X_train_final.iloc[valid_idx].copy(), y_train_full.iloc[valid_idx]
    X_te_test = X_test_final.copy()
    
    # WoE Encoder'ı eğitiyoruz (Regularization ile aşırı öğrenmeyi engelliyoruz)
    woe_encoder = WOEEncoder(cols=cat_cols, randomized=True, random_state=42)
    
    X_tr[cat_cols] = woe_encoder.fit_transform(X_tr[cat_cols], y_tr)
    X_va[cat_cols] = woe_encoder.transform(X_va[cat_cols])
    X_te_test[cat_cols] = woe_encoder.transform(X_te_test[cat_cols])
    
    model = XGBClassifier(
        n_estimators=600,
        learning_rate=0.03,
        max_depth=5, # WoE çok güçlü olduğu için ağacı biraz sığ tutuyoruz (Overfit olmasın diye)
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="auc"
    )
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    
    valid_preds = model.predict_proba(X_va)[:, 1]
    oof_preds[valid_idx] = valid_preds
    test_preds += model.predict_proba(X_te_test)[:, 1] / skf.n_splits
    
    print(f"WoE Fold {fold+1} AUC: {roc_auc_score(y_va, valid_preds):.4f}")

cv_auc = roc_auc_score(y_train_full, oof_preds)
print(f"\n🏆 WoE Encoding (OOF) AUC Skoru: {cv_auc:.4f}")

# Gönderim Dosyası
submission = pd.DataFrame({'id': X_test_final.index, 'Churn': test_preds})
submission.to_csv("../submissions/submission_woe_encoding.csv", index=False)
print("✅ WoE gönderim dosyası kaydedildi!")


--- KAGGLE FORUM TAKTİĞİ: WoE (Weight of Evidence) KODLAMA ---
Sayısal veriler parçalanıyor (QCut Binning)...
Toplam 25 sütuna WoE Encode uygulanacak. Eğitim Başlıyor...

WoE Fold 1 AUC: 0.9137
WoE Fold 2 AUC: 0.9139
WoE Fold 3 AUC: 0.9118
WoE Fold 4 AUC: 0.9130
WoE Fold 5 AUC: 0.9139

🏆 WoE Encoding (OOF) AUC Skoru: 0.9132
✅ WoE gönderim dosyası kaydedildi!
